# Artificial Neural Networks

Forecast of TSLA´s closing price. 

In [2]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor

tsla = pd.read_csv("tsla_1.csv")
tsla["Date"] = pd.to_datetime(tsla["Date"])
tsla = tsla.set_index("Date").sort_index()

precios = tsla["Close/Last"].str.replace("$", "", regex=False).astype(float)
precios.tail()

Date
2026-08-31    367.950
2026-09-01    356.090
2026-09-02    357.010
2026-09-03    376.365
2026-09-04    354.080
Name: Close/Last, dtype: float64

## Model 1

An input and 10 hidden neurons. Train with the first 200 observations and forecast day 202 of the dataset, not the price of tomorrow.

In [3]:
data = precios.copy()

# Save the original values before normalizing.
data_original = data.copy()

# Normalize the data
media = data.mean()
desviacion = data.std()
data = (data - media) / desviacion

# Use the first 200 observations for X and the next value for y
X_train = data.values[0:200].reshape(-1, 1)
y_train = data.values[1:201]

model = MLPRegressor(hidden_layer_sizes=(10,), max_iter=500, random_state=1)
model.fit(X_train, y_train)

# Forecast and denormalize the prediction for the 202nd observation
prediccion = model.predict(data.values[201].reshape(1, -1)) * desviacion + media

print("Prediccion:", round(prediccion[0], 2))
print("Valor real:", round(data_original.values[202], 2))

Prediccion: 246.53
Valor real: 244.92


## Model 2

SMA of 20 and 50 days are added. The network goes through 20 neurons and it is where the closing price is forecasted for the ending of the file.

In [4]:
data = pd.DataFrame({"Cierre": precios})

# Add the SMA variables as input features
data["SMA_20"] = data["Cierre"].rolling(20).mean()
data["SMA_50"] = data["Cierre"].rolling(50).mean()
data = data.dropna()

data_original = data.copy()

media = data.mean()
desviacion = data.std()
data = (data - media) / desviacion

n = len(data)
X_train = data.values[0:n-1, :]
y_train = data.values[1:n, 0]

model = MLPRegressor(hidden_layer_sizes=(20,), max_iter=1000, random_state=1)
model.fit(X_train, y_train)

prediccion = model.predict(data.values[n-1:n, :]) * desviacion["Cierre"] + media["Cierre"]

print("Prediccion del proximo cierre:", round(prediccion[0], 2))
print("Ultimo cierre observado      :", round(data_original["Cierre"].iloc[-1], 2))

Prediccion del proximo cierre: 351.18
Ultimo cierre observado      : 354.08
